In [1]:
import os
import pandas as pd
import numpy as np
import torch
import json

os.chdir('/work1/talati/zhaoyah/TQP-Vortex')

with open('config.json', 'r') as f:
    config = json.load(f)

epoch = pd.Timestamp('1990-01-01')

In [2]:
lineitem_columns = [
          "L_ORDERKEY",         # identifier
           "L_PARTKEY",         # identifier
           "L_SUPPKEY",         # identifier
           "L_LINENUMBER",      # integer
           "L_QUANTITY",        # decimal
           "L_EXTENDEDPRICE",   # decimal
           "L_DISCOUNT",        # decimal
           "L_TAX",             # decimal
           "L_RETURNFLAG",      # fixed text, size 1
           "L_LINESTATUS",      # fixed text, size 1
           "L_SHIPDATE",        # date
           "L_COMMITDATE",      # date
           "L_RECEIPTDATE",     # date
           "L_SHIPINSTRUCT",    # fixed text, size 25
           "L_SHIPMODE",        # fixed text, size 10
           "L_COMMENT"          # variable text size 44
           ]
lineitem_types = [int, int, int, int, float, float, float, float,
         1, 1, 'date', 'date', 'date', 25, 10, 44]
lineitem_used = ["L_SHIPDATE", "L_RETURNFLAG", "L_LINESTATUS", "L_QUANTITY",
        "L_EXTENDEDPRICE", "L_DISCOUNT", "L_TAX"]

In [3]:
# part
part_columns = [
    "P_PARTKEY",        # identifier
    "P_NAME",           # variable text, size 55
    "P_MFGR",           # fixed text, size 25
    "P_BRAND",          # fixed text, size 10
    "P_TYPE",           # variable text, size 25
    "P_SIZE",           # integer
    "P_CONTAINER",      # fixed text, size 10
    "P_RETAILPRICE",    # decimal
    "P_COMMENT"         # variable text, size 23
]
part_types = [int, 55, 25, 10, 25, int, 10, float, 23]

# supplier
supplier_columns = [
    "S_SUPPKEY",        # identifier
    "S_NAME",           # fixed text, size 25
    "S_ADDRESS",        # variable text, size 40
    "S_NATIONKEY",      # identifier
    "S_PHONE",          # fixed text, size 15
    "S_ACCTBAL",        # decimal
    "S_COMMENT"         # variable text, size 101
]
supplier_types = [int, 25, 40, int, 15, float, 101]

# partsupp
partsupp_columns = [
    "PS_PARTKEY",       # identifier
    "PS_SUPPKEY",       # identifier
    "PS_AVAILQTY",      # integer
    "PS_SUPPLYCOST",    # decimal
    "PS_COMMENT"        # variable text, size 199
]
partsupp_types = [int, int, int, float, 199]

# customer
customer_columns = [
    "C_CUSTKEY",        # identifier
    "C_NAME",           # fixed text, size 25
    "C_ADDRESS",        # variable text, size 40
    "C_NATIONKEY",      # identifier
    "C_PHONE",          # fixed text, size 15
    "C_ACCTBAL",        # decimal
    "C_MKTSEGMENT",     # fixed text, size 10
    "C_COMMENT"         # variable text, size 117
]
customer_types = [int, 25, 40, int, 15, float, 10, 117]

# orders
orders_columns = [
    "O_ORDERKEY",       # identifier
    "O_CUSTKEY",        # identifier
    "O_ORDERSTATUS",    # fixed text, size 1
    "O_TOTALPRICE",     # decimal
    "O_ORDERDATE",      # date
    "O_ORDERPRIORITY",  # fixed text, size 15
    "O_CLERK",          # fixed text, size 15
    "O_SHIPPRIORITY",   # integer
    "O_COMMENT"         # variable text, size 79
]
orders_types = [int, int, 1, float, 'date', 15, 15, int, 79]

# nation
nation_columns = [
    "N_NATIONKEY",      # identifier
    "N_NAME",           # fixed text, size 25
    "N_REGIONKEY",      # identifier
    "N_COMMENT"         # variable text, size 152
]
nation_types = [int, 25, int, 152]

# region
region_columns = [
    "R_REGIONKEY",      # identifier
    "R_NAME",           # fixed text, size 25
    "R_COMMENT"         # variable text, size 152
]
region_types = [int, 25, 152]

In [ ]:

def produce(SF, name, columns, types, used):
    
    print(f"SF = {SF}")
    file_path = os.path.join(config.get('tables path'), f'{name}-{SF}.tbl')
    print(f"Reading {file_path}")

    df = pd.read_csv(file_path, sep=',', header=None, dtype=str)
    df.columns = columns
    print(df.head())

    def str_to_tensor(s, length):
        encoded = torch.tensor([ord(c) for c in s[:length]], dtype=torch.int8)
        padded = torch.nn.functional.pad(encoded, (0, length - len(encoded)))
        return padded

    for col, ty in zip(columns, types):

        if col.upper().endswith('COMMENT') and col[0] not in ['S', 'C', 'O']:
            continue
        
        print (f"Processing [{col}]")
        if isinstance(ty, int):
            lim = ty
            if lim == 1:
                tensor_data = torch.tensor(df[col].apply(lambda s: ord(s)).values, dtype=torch.int8)
            else:
                tensor_col = []
                for i in range(lim):
                    print (f"Ch {i+1}/{lim}")
                    tensor_col.append(torch.tensor(df[col].apply(lambda s: (ord(s[i]) if i < len(s) else 0)).values, dtype=torch.int8).unsqueeze(1))

                tensor_data = torch.cat(tensor_col, dim=1)
        elif ty == int:
            tensor_data = df[col].astype(int).values
        elif ty == float:
            tensor_data = df[col].astype(float).values
        elif ty == 'date':
            df[col] = pd.to_datetime(df[col])
            tensor_data = ((df[col] - epoch).dt.total_seconds() * 1e9).astype(int).values
        
        tensor = torch.tensor(tensor_data)
        print (f"Saving [{col}] (ROWS = {tensor.shape})")
        tensor_path = os.path.join(config.get('tensors path'), f'SF{SF}-tensor-{col}.pth')
        torch.save(tensor, tensor_path)
        del tensor, tensor_data
    

In [5]:
torches = {}

SF = 100

# produce(SF, "nation", nation_columns, nation_types, [])
# produce(SF, "region", region_columns, region_types, [])
# produce(SF, "supplier", supplier_columns, supplier_types, [])
produce(SF, "customer", customer_columns, customer_types, [])
produce(SF, "part", part_columns, part_types, [])
produce(SF, "orders", orders_columns, orders_types, [])
produce(SF,  "partsupp", partsupp_columns, partsupp_types, [])
produce(SF, "lineitem", lineitem_columns, lineitem_types, [])

SF = 100
Reading /work1/talati/zhaoyah/tpch-dbgen/customer-100.tbl
  C_CUSTKEY              C_NAME                       C_ADDRESS C_NATIONKEY  \
0         1  Customer#000000001               j5JsirBM9PsCy0O1m          15   
1         2  Customer#000000002  487LW1dovn6Q4dMVymKwwLE9OKf3QG          13   
2         3  Customer#000000003                    fkRGN8nY4pkE           1   
3         4  Customer#000000004                     4u58h fqkyE           4   
4         5  Customer#000000005    hwBtxkoBF qSW4KrIk5U 2B1AU7H           3   

           C_PHONE C_ACCTBAL C_MKTSEGMENT  \
0  25-989-741-2988    711.56     BUILDING   
1  23-768-687-3665    121.65   AUTOMOBILE   
2  11-719-748-3364   7498.12   AUTOMOBILE   
3  14-128-190-5944   2866.83    MACHINERY   
4  13-750-942-6364    794.47    HOUSEHOLD   

                                           C_COMMENT  
0  regular, regular platelets are fluffily accord...  
1  furiously special deposits solve slyly. furiou...  
2                   sp

/tmp/ipykernel_379711/3783282254.py:41: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  tensor = torch.tensor(tensor_data)


Saving [C_NAME] (ROWS = torch.Size([15000000, 25]))
Processing [C_ADDRESS]
Ch 1/40


MemoryError: Unable to allocate 229. MiB for an array with shape (15000000,) and data type complex128